In [1]:
# 0. KERNEL CLEANER & PERFECT LZMA MOCK (Prevents DLL Crash)
import sys
import types

for k in list(sys.modules.keys()):
    if k.startswith("datasets") or k.startswith("fsspec") or k.startswith("sklearn") or k == "lzma" or k == "_lzma":
        del sys.modules[k]

fake_lzma = types.ModuleType("lzma")
fake_lzma.LZMAError = Exception
class FakeLZMAFile:
    def __init__(self, *args, **kwargs): pass
    def read(self, *args, **kwargs): return b""
    def write(self, *args, **kwargs): return 0
    def seek(self, *args, **kwargs): return 0
    def tell(self, *args, **kwargs): return 0
    def close(self): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
fake_lzma.LZMAFile = FakeLZMAFile
fake_lzma.open = lambda *args, **kwargs: FakeLZMAFile()
fake_lzma.is_check_supported = lambda check: False
fake_lzma.FORMAT_AUTO, fake_lzma.FORMAT_XZ, fake_lzma.FORMAT_ALONE, fake_lzma.FORMAT_RAW = 0, 1, 2, 3
fake_lzma.CHECK_NONE, fake_lzma.CHECK_CRC32, fake_lzma.CHECK_CRC64, fake_lzma.CHECK_SHA256 = 0, 1, 4, 10
sys.modules["lzma"], sys.modules["_lzma"] = fake_lzma, fake_lzma
sys.modules["torchvision"] = None 

# 1. IMPORTS & SETUP
import os
import random
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_from_disk
from sklearn.metrics import roc_curve, auc, confusion_matrix
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

BASE_DIR = r"E:\SIH2026\models"
PLOTS_DIR = os.path.join(BASE_DIR, "calibration_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Evaluation Device: {device.upper()}")

TARGET_LENGTH = 48000
BATCH_SIZE = 16
SAMPLES_PER_LANG = 1000

LANG_CONFIGS = [
    {"lang": "Hindi", "tts_dir": "hindi_indicTTS", "synth_dir": "hindi_indicsynth", "model_dir": "trained_combined_hindi"},
    {"lang": "Bengali", "tts_dir": "bengali_indicTTS", "synth_dir": "bengali_indicsynth", "model_dir": "trained_combined_bengali"},
    {"lang": "Telugu", "tts_dir": "telugu_indicTTS", "synth_dir": "telugu_indicsynth", "model_dir": "trained_combined_telugu"}
]

def calculate_prosodic_anomaly(audio_array, sampling_rate=16000):
    try:
        frame_len, hop_len = int(0.025 * sampling_rate), int(0.010 * sampling_rate)
        frames = np.array([audio_array[i : i + frame_len] for i in range(0, len(audio_array) - frame_len, hop_len)])
        if len(frames) == 0: return 0.5
        ste = np.sum(frames**2, axis=1)
        ste = ste / (np.max(ste) + 1e-6)
        return float(min(max(np.exp(-45 * np.var(ste)), 0.0), 1.0))
    except: return 0.5

def compute_hybrid_spoof_score(w2v_prob: float, prosody_val: float) -> float:
    fused_score = max(w2v_prob, 0.70 + (prosody_val * 0.25)) if prosody_val > 0.40 else (0.65 * w2v_prob) + (0.35 * prosody_val)
    return float(min(max(fused_score, 0.0), 1.0))

def load_audio_file(file_path):
    data, sr = sf.read(file_path, dtype='float32')
    if data.ndim > 1: data = np.mean(data, axis=1)
    if sr != 16000:
        import librosa
        data = librosa.resample(data, orig_sr=sr, target_sr=16000)
    return data

def fetch_combined_data(tts_path, synth_path, quota_real=250, quota_fake=250):
    b_audios, b_prosody, s_audios, s_prosody = [], [], [], []
    
    # IndicTTS
    if os.path.exists(tts_path):
        tts_real, tts_fake = [], []
        for root, _, files in os.walk(tts_path):
            for f in files:
                if f.lower().endswith(('.wav', '.mp3', '.flac')):
                    p = os.path.join(root, f)
                    tts_fake.append(p) if "fake" in p.lower() or "spoof" in p.lower() else tts_real.append(p)
        random.seed(42); random.shuffle(tts_real); random.shuffle(tts_fake)
        for p in tts_real[:quota_real]:
            try: arr = load_audio_file(p); b_audios.append(arr); b_prosody.append(calculate_prosodic_anomaly(arr))
            except: pass
        for p in tts_fake[:quota_fake]:
            try: arr = load_audio_file(p); s_audios.append(arr); s_prosody.append(calculate_prosodic_anomaly(arr))
            except: pass

    # IndicSynth
    synth_val = os.path.join(synth_path, "validation")
    if os.path.exists(synth_val):
        try:
            ds = load_from_disk(synth_val)
            fake_val = 1
            if "label" in ds.features and hasattr(ds.features["label"], "names"):
                names = [n.lower() for n in ds.features["label"].names]
                if "fake" in names: fake_val = names.index("fake")
            r_c, f_c = 0, 0
            for row in ds:
                is_fake = (row.get("label", row.get("is_tts", 0)) == fake_val)
                raw = np.array(row["audio"]["array"], dtype=np.float32)
                if not is_fake and r_c < quota_real: b_audios.append(raw); b_prosody.append(calculate_prosodic_anomaly(raw)); r_c += 1
                elif is_fake and f_c < quota_fake: s_audios.append(raw); s_prosody.append(calculate_prosodic_anomaly(raw)); f_c += 1
                if r_c >= quota_real and f_c >= quota_fake: break
        except Exception as e: print(f"Warning: {e}")

    return b_audios, b_prosody, s_audios, s_prosody

# 5. EXECUTE BENCHMARK LOOP WITH PER-LANGUAGE CALIBRATION TABLE
summary_records = []

for config in LANG_CONFIGS:
    lang = config["lang"]
    tts, synth, model_dir = os.path.join(BASE_DIR, config["tts_dir"]), os.path.join(BASE_DIR, config["synth_dir"]), os.path.join(BASE_DIR, config["model_dir"])
    
    if not os.path.exists(model_dir): continue
    
    print("\n" + "="*85)
    print(f"BENCHMARKING & CALIBRATING: {lang.upper()}")
    print("="*85)

    extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_dir)
    model = Wav2Vec2ForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()

    b_aud, b_pros, s_aud, s_pros = fetch_combined_data(tts, synth, 250, 250)
    y_true = np.array([0]*len(b_aud) + [1]*len(s_aud))
    all_raw = b_aud + s_aud
    
    proc_tensors = []
    for wave in all_raw:
        if len(wave) > TARGET_LENGTH: proc_tensors.append(wave[(len(wave)-TARGET_LENGTH)//2 : (len(wave)+TARGET_LENGTH)//2])
        else: proc_tensors.append(np.pad(wave, (0, TARGET_LENGTH - len(wave)), 'constant'))

    raw_spoof = []
    for i in range(0, len(proc_tensors), BATCH_SIZE):
        batch = proc_tensors[i:i+BATCH_SIZE]
        inputs = extractor(batch, sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1).cpu().numpy()
        raw_spoof.extend(probs[:, 1].tolist())

    fused_scores = [compute_hybrid_spoof_score(w, p) for w, p in zip(raw_spoof, b_pros + s_pros)]
    y_scores = np.array(fused_scores)
    
    # PRINT CALIBRATION TABLE
    print(f"\n{'-'*80}")
    print(f"{'Threshold':<10} | {'Bonafide Acc (Real)':<22} | {'Spoof Acc (Fake)':<20} | {'Overall Acc':<12} | {'EER Gap':<10}")
    print(f"{'-'*80}")

    best_thresh, min_gap = 0.50, 100.0
    best_stats = {}
    
    for th in np.arange(0.40, 0.99, 0.03):
        preds = (y_scores >= th).astype(int)
        bf_acc = (preds[y_true == 0] == 0).mean() * 100
        sp_acc = (preds[y_true == 1] == 1).mean() * 100
        tot_acc = (preds == y_true).mean() * 100
        gap = abs(bf_acc - sp_acc)
        
        print(f"{th:<10.2f} | {bf_acc:<20.2f}% | {sp_acc:<18.2f}% | {tot_acc:<10.2f}% | {gap:<8.2f}%")
        
        if gap < min_gap:
            min_gap, best_thresh = gap, th
            best_stats = {"bf_acc": bf_acc, "sp_acc": sp_acc, "ov_acc": tot_acc}

    # Print Averages
    avg_real_fused = float(np.mean(y_scores[y_true == 0]))
    avg_fake_fused = float(np.mean(y_scores[y_true == 1]))
    avg_b_pros = float(np.mean(b_pros)) if b_pros else 0.0
    avg_s_pros = float(np.mean(s_pros)) if s_pros else 0.0
    
    print(f"\nRECOMMENDED CALIBRATED THRESHOLD FOR {lang.upper()}: {best_thresh:.2f}")
    print(f"Expected Balanced Bonafide Accuracy : {best_stats['bf_acc']:.2f}%")
    print(f"Expected Balanced Spoof Accuracy    : {best_stats['sp_acc']:.2f}%")
    print(f"Average Real Score  (Fused): {avg_real_fused:.4f} | Prosody: {avg_b_pros:.4f}")
    print(f"Average Fake Score  (Fused): {avg_fake_fused:.4f} | Prosody: {avg_s_pros:.4f}")
    
    summary_records.append({
        "Language": lang, "Calibrated Thresh": round(best_thresh, 2), "Overall Acc": f"{best_stats['ov_acc']:.2f}%",
        "Avg Real Fused": round(avg_real_fused, 4), "Avg Fake Fused": round(avg_fake_fused, 4)
    })
    
    del model, extractor; torch.cuda.empty_cache()

print("\n" + "="*80)
print(pd.DataFrame(summary_records).to_string(index=False))

Evaluation Device: CUDA

BENCHMARKING & CALIBRATING: HINDI


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
Threshold  | Bonafide Acc (Real)    | Spoof Acc (Fake)     | Overall Acc  | EER Gap   
--------------------------------------------------------------------------------
0.40       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.43       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.46       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.49       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.52       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.55       | 0.00                % | 99.80             % | 66.53     % | 99.80   %
0.58       | 0.40                % | 99.80             % | 66.67     % | 99.40   %
0.61       | 9.20                % | 93.20             % | 65.20     % | 84.00   %
0.64       | 28.80               % | 73.00             % | 58.27     % | 44.20   %
0.6

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
Threshold  | Bonafide Acc (Real)    | Spoof Acc (Fake)     | Overall Acc  | EER Gap   
--------------------------------------------------------------------------------
0.40       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.43       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.46       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.49       | 0.40                % | 99.80             % | 66.67     % | 99.40   %
0.52       | 2.40                % | 98.60             % | 66.53     % | 96.20   %
0.55       | 11.60               % | 92.40             % | 65.47     % | 80.80   %
0.58       | 24.00               % | 74.40             % | 57.60     % | 50.40   %
0.61       | 38.80               % | 56.60             % | 50.67     % | 17.80   %
0.64       | 51.60               % | 39.80             % | 43.73     % | 11.80   %
0.6

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]


--------------------------------------------------------------------------------
Threshold  | Bonafide Acc (Real)    | Spoof Acc (Fake)     | Overall Acc  | EER Gap   
--------------------------------------------------------------------------------
0.40       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.43       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.46       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.49       | 0.00                % | 100.00            % | 66.67     % | 100.00  %
0.52       | 2.40                % | 98.60             % | 66.53     % | 96.20   %
0.55       | 9.60                % | 90.80             % | 63.73     % | 81.20   %
0.58       | 24.00               % | 70.60             % | 55.07     % | 46.60   %
0.61       | 46.40               % | 46.80             % | 46.67     % | 0.40    %
0.64       | 58.40               % | 37.40             % | 44.40     % | 21.00   %
0.6